### Imports

In [1]:
import cv2
import numpy as np
import glob
import time

# 1. Record Photos

In [2]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# prepare object points, like (0,0,0), (1,0,0), (2,0,0) ....,(6,5,0)
objp = np.zeros((10*7,3), np.float32)
objp[:,:2] = np.mgrid[0:10,0:7].T.reshape(-1,2)
# compensate for square size
square_size = 0.022
objp *= square_size
 
# Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.

In [ ]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if (not ret): continue

    key = cv2.waitKey(1)

    if key == ord(' '):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        ret, corners = cv2.findChessboardCorners(gray, (10,7), None)

        if ret:
            objpoints.append(objp)

            corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
            imgpoints.append(corners2)

            cv2.imwrite("images/calib_%f.jpg" % time.time(), frame)
            cv2.drawChessboardCorners(frame, (10,7), corners2, ret)

    cv2.imshow("Calibration", frame)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

# 2. Compute points from photos

In [3]:
images = glob.glob('images/*.jpg')

print(len(images))

for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
 
    # Find the chess board corners
    ret, corners = cv2.findChessboardCorners(gray, (10,7), None)
 
    # If found, add object points, image points (after refining them)
    if ret:
        objpoints.append(objp)
 
        corners2 = cv2.cornerSubPix(gray,corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners2)

12


# 3. Run Calibration

In [4]:
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)

In [12]:
np.savetxt('calib_mtx.csv', mtx, delimiter=',')
np.savetxt('calib_dist.csv', dist, delimiter=',')

In [16]:
fmtx = np.array(np.loadtxt('calib_mtx.csv', delimiter=','))

## 3.1 Optimize Matrix

In [5]:
img = cv2.imread(images[0])
h, w = img.shape[:2]
newcammtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))

## 3.2 Test undistortion

In [6]:
undistort = False
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if (not ret): continue

    if (undistort):
        frame = cv2.undistort(frame, mtx, dist, None, newcammtx)
        # x, y, w, h = roi
        # frame = frame[]
    
    cv2.imshow("Camera", frame)

    key = cv2.waitKey(1)

    if key == ord('u'):
        undistort = not undistort
    elif key == ord('q'):
        break

cap.release()

cv2.destroyAllWindows()
cv2.waitKey(1)

-1

# 4. Load AR

In [7]:
aruco_dict = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_50
)

# detector = cv2.aruco.ArucoDetector(aruco_dict)
# cv2.aruco.detectMarkers()

# 5. Run Detection

In [8]:
cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

def drawSquare(frame, corners):
    for i in range(4):
        cv2.line(frame, (int(corners[0][i][0]),int(corners[0][i][1])), (int(corners[0][(i+1)%4][0]), int(corners[0][(i+1)%4][1])), (0,0,255), 3)

tagSize = 38 / 1000 # m

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if (not ret): continue

    corners, ids, _ = cv2.aruco.detectMarkers(frame, aruco_dict)

    objpoints = np.array([
        (-tagSize/2, tagSize/2, 0),
        (tagSize/2, tagSize/2, 0),
        (tagSize/2, -tagSize/2, 0),
        (-tagSize/2, -tagSize/2, 0)
    ])

    rvecs = []
    tvecs = []
    
    if corners is not None and ids is not None:
        for square, id in zip(corners, ids):
            drawSquare(frame, square)
            ret, rvec, tvec = cv2.solvePnP(objpoints, square[0], mtx, dist)
            rvecs.append(rvec)
            tvecs.append(tvec)
            distance = np.linalg.norm(tvec) 
            cv2.putText(frame, '%f mm' % (distance * 1000), (int(square[0][0][0]), int(square[0][0][1])), cv2.FONT_HERSHEY_COMPLEX, 1, (60,255,70), 2)
            cv2.drawFrameAxes(frame, mtx, dist, rvec, tvec, tagSize * 1.5, 2)

            # TODO: calculate location of centroid based on dimensions of block and pose vector
            # Centroid is (block length / 2) along block's pose vectors rx, ry, rz

    cv2.imshow('ArUco', frame)

    key = cv2.waitKey(1)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)


-1

# 6 Transforms

In [1]:
import cv2
import numpy as np
import glob
import time
from pymycobot.mycobot import MyCobot
from pymycobot import PI_PORT, PI_BAUD

mtx = np.array(np.loadtxt('calib_mtx.csv', delimiter=','))
dist = np.array(np.loadtxt('calib_dist.csv', delimiter=','))

aruco_dict = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_50
)

tagSize = 38 / 1000

In [ ]:

cap = cv2.VideoCapture(0)
START_POS = [0,0,0,0,0,0]
NEUTRAL = [0, -80, 65, 0, 45, 0]

bot = MyCobot(PI_PORT, PI_BAUD)
bot.send_angles(NEUTRAL, 40)
time.sleep(2)
bot.set_gripper_value(95, 40, 1)
time.sleep(2)
bot.send_angles(NEUTRAL, 40)
time.sleep(2)

while True:
    ret, frame = cap.read()
    if (not ret): continue

    key = cv2.waitKey(1)

    if key == ord(' '):
        corners, ids, _ = cv2.aruco.detectMarkers(frame, aruco_dict)
        corner = corners[0]
        id = ids[0]

        objpoints = np.array([
            (-tagSize/2, tagSize/2, 0),
            (tagSize/2, tagSize/2, 0),
            (tagSize/2, -tagSize/2, 0),
            (-tagSize/2, -tagSize/2, 0)
        ])

        # try with identity matrices (?)
        # units?? 
        ret, rvec, tvec = cv2.solvePnP(objpoints, corner, mtx, dist)

        # Need rot & transform from camera to ee
        rot = np.array([
            [0, 0, 1],
            [0, 1, 0],
            [-1, 0, 0]
        ]) @ np.array([
            [0, -1, 0],
            [1, 0, 0],
            [0, 0, 1]
        ])

        block_center = rot @ tvec
        block_center[0] += tagSize/2

        desired_pose = block_center
        desired_pose[2] += 1.5 * tagSize

        print(desired_pose)

        # bot.send_coords(desired_pose, 30)
        # time.sleep(2)

    cv2.imshow('Transform', frame)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

Note: This class is no longer maintained since v3.6.0, please refer to the project documentation: https://github.com/elephantrobotics/pymycobot/blob/main/README.md
[[0.20393075]
 [0.02825431]
 [0.12984313]]


-1

In [4]:
from pymycobot.mycobot import MyCobot
from pymycobot import PI_PORT, PI_BAUD
import time

START_POS = [0,0,0,0,0,0]
NEUTRAL = [0, -60, 65, 0, 45, 0]

bot = MyCobot(PI_PORT, PI_BAUD)
bot.send_angles(START_POS, 40)
time.sleep(2)
bot.set_gripper_value(95, 40, 1)
time.sleep(2)
bot.send_angles(NEUTRAL, 40)
time.sleep(2)

Note: This class is no longer maintained since v3.6.0, please refer to the project documentation: https://github.com/elephantrobotics/pymycobot/blob/main/README.md
